In [1]:
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm

# Import RAG

In [2]:
import sys
sys.path.append('../scripts')
import rag
import vectors

# Load synthetic data

In [3]:
df_synth = pd.read_csv('../data/data-synth-question.csv', sep='\t', dtype=str)
df_synth

,pmid,ollama_seed,synthetic_question
0,40247608,0,What causes a significant portion of individua...
1,40247608,1,Is Fragile X syndrome considered a type of aut...
2,40247608,2,What causes some individuals with autism to ex...
3,40247608,3,What causes the unique communication styles an...
4,40247608,4,What are the main differences between Fragile ...
...,...,...,...
495,40933686,0,What challenges do autistic individuals face w...
496,40933686,1,Can we do away with the stigma surrounding aut...
497,40933686,2,What strategies do universities need to implem...
498,40933686,3,What challenges do autistic individuals face w...


# Demo retriever

In [4]:
# select question to demonstrate
demo_query = df_synth.iloc[0]['synthetic_question']
print(demo_query)

What causes a significant portion of individuals with autism spectrum disorders to have Fragile X syndrome as their underlying genetic condition?


In [5]:
# demonstrate vector search
rag.vsearch(demo_query, num_results=2)

[{'pmid': '40299377',
  'elocationid': 'pii: 805',
  'title': 'From Discovery to Innovative Translational Approaches in 80 Years of Fragile X Syndrome Research.',
  'journal': 'Biomedicines',
  'year': '2025',
  'author': 'Mathijs B van der Lei, R Frank Kooy',
  'affiliation': 'Center of Medical Genetics, University of Antwerp and Antwerp University Hospital, 2650 Edegem, Belgium.',
  'abstract': 'Fragile X syndrome (FXS) is the most common inherited cause of intellectual disability and a major genetic contributor to autism spectrum disorder. It is caused by a CGG trinucleotide repeat expansion in the '},
 {'pmid': '40869951',
  'elocationid': 'pii: 903',
  'title': '',
  'journal': 'Genes',
  'year': '2025',
  'author': 'Maria Dobre, Gisela Gaina, Alina Erbescu, Adelina Glangher, Florentina Ionela Linca, Doina Ioana, Emilia Maria Severin, Florina Rad, Mihaela Catrinel Iliescu, Sorina Mihaela Papuc, Mihail Eugen Hinescu, Aurora Arghir, Magdalena Budișteanu',
  'affiliation': 'Prof. Dr.

In [6]:
# demonstrate keyword search
rag.kwsearch(demo_query, num_results=2)

[{'pmid': '40821657',
  'elocationid': 'pii: 102489',
  'title': 'Functional upper-extremity movements in autism: A narrative literature review.',
  'journal': 'Research in autism spectrum disorders',
  'year': '2024',
  'author': 'Shanan Sun, Nicholas E Fears, Haylie L Miller',
  'affiliation': 'School of Kinesiology, University of Michigan, Ann Arbor, MI, USA.',
  'abstract': 'Many autistic individuals exhibit clinically-significant motor difficulties. Previous reviews focused on overall motor ability or coordination, but with little attention paid to quantifying differences in upper extremity skills, which are critical to many activities of daily living. Our objective was to identify and evaluate the published literature on upper extremity motor skills of autistic people.'},
 {'pmid': '40881177',
  'elocationid': 'pii: 102460',
  'title': 'Exploring the effects of age and sex on sensory sensitivities in middle and older aged autistic adults.',
  'journal': 'Research in autism spectr

# Evaluate retriever, by hit rate and mean reciprocal rank (MRR)

## Define functions

In [7]:
def get_relevance_total(synth_records, search_function, num_results):
    # initialize relevance total
    relevance_total = []
    # for each document, make relevance
    for record in tqdm(synth_records):
        query = record['synthetic_question']
        true_pmid = record['pmid']
        search_results = search_function(query, num_results)
        relevance = [true_pmid==d['pmid'] for d in search_results]
        relevance_total.append(relevance)
    return relevance_total

In [8]:
def hit_rate(relevance_total):
    hit_boolean = [True in line for line in relevance_total]
    numerator = sum(hit_boolean)
    denominator = len(hit_boolean)
    return numerator/denominator

In [9]:
def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1/(rank+1)
    return total_score/len(relevance_total)

In [10]:
def get_questions_by_correctness(synth_records, relevance_total):
    questions_answered_correct = []
    questions_answered_incorrect = []
    for i in range(len(synth_records)):
        relevance = relevance_total[i]
        question = synth_records[i]['synthetic_question']
        if True in relevance:
            questions_answered_correct.append(question)
        else:
            questions_answered_incorrect.append(question)
    all_questions = {'correct' : questions_answered_correct, \
    'incorrect' : questions_answered_incorrect}
    return all_questions

## Work

In [11]:
# cast as list of dictionaries
synth_records = df_synth.to_dict('records')
print(len(synth_records))

500


In [12]:
# number of documents retrieved
num_results = rag.config['num_results']
print(num_results)

5


In [13]:
# for vector search, get relevance total
print(datetime.now())
relevance_total_v = get_relevance_total(synth_records=synth_records, \
search_function=rag.vsearch, num_results=num_results)
print(datetime.now())

2025-09-13 02:01:06.513833


  0%|          | 0/500 [00:00<?, ?it/s]

2025-09-13 02:02:07.989238


In [14]:
(lambda x : {'hit rate' : hit_rate(x), 'mrr' : mrr(x)})(relevance_total_v)

{'hit rate': 0.414, 'mrr': 0.3184666666666666}

In [15]:
# for vector search, see questions correctly and incorrectly answered
qbc_v = get_questions_by_correctness(synth_records, relevance_total_v)
print('---correct---')
print('\n'.join(qbc_v['correct'][:10]))
print()
print('---incorrect---')
print('\n'.join(qbc_v['incorrect'][:10]))

---correct---
What causes a significant portion of individuals with autism spectrum disorders to have Fragile X syndrome as their underlying genetic condition?
Is Fragile X syndrome considered a type of autism?
What are the main differences between Fragile X syndrome and other forms of intellectual disability that may also be associated with autism?
What is the underlying mechanism by which bi-allelic UGGT1 variants lead to congenital disorders of glycosylation?
What is the primary difference between individuals with bi-allelic UGGT1 variants in CDGs compared to those without these variants?
What is the underlying cause of congenital disorders of glycosylation (CDGs)?
What is the primary difference between individuals with autism spectrum disorder (ASD) that can be identified through neurocognitive profile analysis?
What causes Fragile X syndrome's unique combination of intellectual disability and social difficulties?
Is FXS typically caused by a single gene mutation, or is it often ac

In [16]:
# for keyword search, get relevance total
print(datetime.now())
relevance_total_kw = get_relevance_total(synth_records=synth_records, \
search_function=rag.kwsearch, num_results=num_results)
print(datetime.now())

2025-09-13 02:02:08.037727


  0%|          | 0/500 [00:00<?, ?it/s]

2025-09-13 02:02:10.348494


In [17]:
(lambda x : {'hit rate' : hit_rate(x), 'mrr' : mrr(x)})(relevance_total_kw)

{'hit rate': 0.45, 'mrr': 0.36249999999999993}

In [18]:
# for keyword search, seee questions correctly and incorrectly answered
qbc_kw = get_questions_by_correctness(synth_records, relevance_total_kw)
print('---correct---')
print('\n'.join(qbc_kw['correct'][:10]))
print()
print('---incorrect---')
print('\n'.join(qbc_kw['incorrect'][:10]))

---correct---
Is Fragile X syndrome considered a type of autism?
What is the underlying mechanism by which bi-allelic UGGT1 variants lead to congenital disorders of glycosylation?
What is the primary difference between individuals with bi-allelic UGGT1 variants in CDGs compared to those without these variants?
What is the underlying cause of congenital disorders of glycosylation (CDGs)?
What is the primary difference between individuals with autism spectrum disorder (ASD) that can be identified through neurocognitive profile analysis?
What causes the genetic mutation that leads to Fragile X syndrome, which is also a significant risk factor for developing autism spectrum disorder?
How does maternal immune activation (MIA) affect the development and functioning of white matter in the brain?
How do changes in gene expression due to congenital adrenal hyperplasia (CAH) affect brain structure and function?
Is autism caused by a specific genetic mutation affecting cortisol and adrenal hormon

In [19]:
print(datetime.now())

2025-09-13 02:02:10.362942
